# Receive public comparison of five existing fixed-score observations

Owner: BindingMT #78. **Prospective, unexecuted qualification.** Freeze/register
this notebook, cells, prescription, runtime and inputs before execution.

Reproduce five preparations from stored full18 chemical/structural models with
five explicit new Gasteiger–Marsili and AutoDock4 chemical-environment
assignments. Generate no H and compute no energy. This is new preparation
evidence, not restoration of historical MM or historical prepared-H identity.

Receive Vina and Vinardo separately under normal and pm/fs policies through
public `compare_scores`. Require exact reproduction of captured input bytes
first. Instrumentation and existing inspection helpers are qualification only.
The two frozen roots are trusted solely for this identified population.

The supervisor must enforce CPU 0, nice 10, one worker, an 8 GiB available-RAM
reserve, 1 GiB owned RSS, 2 GiB address space, 60 CPU seconds and 90 wall seconds.
Set `BINDINGMT_COMPARISON_ROOT` and fresh `BINDINGMT_COMPARISON_OUTPUT`.
Preserve failures and partial outputs; never retry automatically.


In [ ]:
import base64
import gzip
import hashlib
import json
import os
import sys
from copy import copy, deepcopy
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

root = Path(os.environ['BINDINGMT_COMPARISON_ROOT']).resolve()
output = Path(os.environ['BINDINGMT_COMPARISON_OUTPUT']).resolve()
assert not output.exists() and not root.is_relative_to(output)
assert os.sched_getaffinity(0) == {0} and os.getpriority(os.PRIO_PROCESS, 0) >= 10
output.mkdir(parents=True, exist_ok=False)
ROOTS = {
    'construction': '45b3665ba09b6979aa1bcdf1bea0ac3e8102f081fefbfc51348c55bc71405df3',
    'scoring': 'bf214aa7e48f15a95051411ed3bd54969f78c811503050c6969d6f5aa2898e93',
}
directories = {
    'construction': root / 'devguide/validation/data/e1_complete_pose_2026-11-06',
    'scoring': root / 'devguide/validation/data/e1_native_rescoring_2026-11-06',
}


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def safe_relative(name):
    value = PurePosixPath(name)
    assert not value.is_absolute() and '..' not in value.parts and str(value) == name
    return name


manifests, stored_inputs = {}, {}
for stage, directory in directories.items():
    manifest_path = directory / 'manifest.json'
    assert file_sha(manifest_path) == ROOTS[stage]
    manifest = json.loads(manifest_path.read_text())
    assert manifest['owner'] == 'uibcdf/bindingmt#78'
    manifests[stage] = manifest
    stored_inputs[str(manifest_path)] = ROOTS[stage]
    for row in manifest['files']:
        path = directory / safe_relative(row['path'])
        assert path.resolve().is_relative_to(directory.resolve())
        data = path.read_bytes()
        assert len(data) == row['bytes']
        assert hashlib.sha256(data).hexdigest() == row['sha256']
        stored_inputs[str(path)] = row['sha256']
        if row['path'].endswith('.gz'):
            decoded = gzip.decompress(data)
            assert len(decoded) == row['decoded_bytes']
            assert hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']


def input_bytes(stage, name):
    rows = [
        r
        for r in manifests[stage]['files']
        if r['path'] == name or r.get('original_path') == name
    ]
    assert len(rows) == 1, name
    row = rows[0]
    data = (directories[stage] / row['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['sha256']
    return gzip.decompress(data) if row['path'].endswith('.gz') else data


collection_record = json.loads(input_bytes('scoring', 'scored_collection.json'))
contract = json.loads(input_bytes('construction', 'prospective_scoring_contract.json'))
native_receipt = json.loads(input_bytes('scoring', 'receipt.json'))
assert native_receipt['input_sha256']['manifest'] == ROOTS['construction']
assert len(collection_record['poses']) == 5
assert [r['pose_id'] for r in collection_record['poses']] == [
    f'e1-new-local-H-{i}' for i in range(1, 6)
]
inputs = output / 'inputs'
inputs.mkdir()
receptor_path = inputs / 'provided_receptor.pdbqt'
receptor_path.write_bytes(input_bytes('construction', 'provided_receptor.pdbqt'))
assert file_sha(receptor_path) == contract['receptor_sha256']
source_paths = []
for index in range(1, 6):
    path = inputs / f'full18_pose_{index}.h5msm'
    path.write_bytes(
        input_bytes('construction', f'normal/pose_{index}/full18_chemical_only.h5msm')
    )
    source_paths.append(path)
local_inputs_before = {
    str(path): file_sha(path) for path in [receptor_path, *source_paths]
}


class RejectOptionalImports:
    def find_spec(self, fullname, path=None, target=None):
        if fullname.split('.')[0] in {'vina', 'molsysviewer'}:
            raise AssertionError('Forbidden optional import: ' + fullname)
        return None


assert not any(name.split('.')[0] in {'vina', 'molsysviewer'} for name in sys.modules)
import_guard = RejectOptionalImports()
sys.meta_path.insert(0, import_guard)

In [ ]:
import importlib.metadata as metadata
from functools import wraps

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
from smonitor.integrations import CatalogException

import bindingmt as dmt
import bindingmt.comparison as comparison_api
import bindingmt.engines._vina_scoring as scoring_adapter
import bindingmt.preparation._export as prepared_export
from bmt_test_support.qualify_chemical_templates import snapshot as chemical_snapshot

budgets = dict(
    source_loads=5,
    prepare_ligand=5,
    partial_charge_assignment=5,
    atom_type_assignment=5,
    comparison_attempts=12,
    comparison_returns=4,
    expected_refusals=8,
    exports=26,
    gate_exports=5,
    comparison_exports=21,
    comparison_extracts=20,
    chemical_correspondences=20,
    comparison_charge_audits=20,
    selected_evaluation_audits=35,
    geometry_reads=20,
    inspection_snapshots=30,
    detachment_controls=4,
    json_roundtrips=4,
    changed_preparation_controls=1,
)
counts = dict.fromkeys(budgets, 0)
phase = 'setup'
caller_policy = puw.configure.report()
patches, forbidden_attempts, export_journal = [], [], []


def plain(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    raise TypeError('Unsupported inspection value: ' + type(value).__name__)


def normalized(value):
    return json.loads(json.dumps(value, default=plain, allow_nan=False))


def save(path, value):
    with path.open('x', encoding='utf-8') as stream:
        json.dump(value, stream, default=plain, indent=2, allow_nan=False)
        stream.write('\n')


def event(kind, **details):
    with (output / 'operations.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(
            json.dumps(
                dict(
                    kind=kind,
                    at_utc=datetime.now(timezone.utc).isoformat(),
                    counts=dict(counts),
                    **details,
                ),
                allow_nan=False,
            )
            + '\n'
        )


def consume(name):
    assert counts[name] < budgets[name], 'Operation cap reached: ' + name
    counts[name] += 1


def replace(owner, name, value):
    patches.append((owner, name, getattr(owner, name)))
    setattr(owner, name, value)


def counted(function, name):
    @wraps(function)
    def wrapped(*args, **kwargs):
        consume(name)
        event('started', operation=name)
        result = function(*args, **kwargs)
        event('returned', operation=name)
        return result

    return wrapped


def forbidden(name):
    def stopped(*args, **kwargs):
        forbidden_attempts.append(name)
        event('forbidden_attempt', operation=name)
        raise AssertionError('Forbidden operation: ' + name)

    return stopped


def native_inspection(system):
    consume('inspection_snapshots')
    # Existing qualification helper uses public owner domains; no restoration.
    return normalized(
        dict(
            chemistry_structures=chemical_snapshot(system),
            mechanics=system.molecular_mechanics.to_dict(),
            native_charge_unit='elementary_charge',
            charge_assignment=system.molecular_mechanics.partial_charge_assignment,
            type_assignment=system.molecular_mechanics.atom_type_assignment,
        )
    )


def prepared_fields(prepared):
    return normalized(
        dict(
            declaration=prepared.to_dict(),
            coordinates_nm=puw.get_value(prepared.coordinates, to_unit='nm'),
        )
    )


assert dmt.compare_scores is comparison_api.compare_scores
assert file_sha(Path(comparison_api.__file__)) == file_sha(
    root / 'bindingmt/comparison.py'
)
identities = {
    name: dict(
        version=metadata.version(name),
        path=str(Path(module.__file__).resolve()),
        sha256=file_sha(Path(module.__file__)),
    )
    for name, module in [('bindingmt', dmt), ('molsysmt', msm), ('pyunitwizard', puw)]
}
identities['comparison_source'] = dict(
    path=str(Path(comparison_api.__file__).resolve()),
    sha256=file_sha(Path(comparison_api.__file__)),
)
identities['inspection_helper'] = dict(
    path=str(Path(chemical_snapshot.__code__.co_filename).resolve()),
    sha256=file_sha(Path(chemical_snapshot.__code__.co_filename)),
)
assert identities['inspection_helper']['sha256'] == file_sha(
    root / 'bmt_test_support/qualify_chemical_templates.py'
)
native_extension = Path(msm.__file__).parent / '_rust.abi3.so'
identities['native_extension'] = dict(
    path=str(native_extension.resolve()), sha256=file_sha(native_extension)
)
identities['dependency_versions'] = {
    name: metadata.version(name)
    for name in ('numpy', 'rdkit', 'argdigest', 'depdigest', 'smonitor')
}
save(output / 'runtime_identity.json', identities)
original_export = prepared_export.export_pdbqt


@wraps(original_export)
def monitored_export(*args, **kwargs):
    consume('exports')
    consume('gate_exports' if phase == 'gate' else 'comparison_exports')
    result = original_export(*args, **kwargs)
    item = normalized(
        dict(
            sequence=counts['exports'],
            phase=phase,
            sha256=hashlib.sha256(result.text.encode()).hexdigest(),
            provider_atom_map=result.atom_map,
        )
    )
    export_journal.append(item)
    save(output / f'export_{counts["exports"]:02d}.json', item)
    return result


original_extract = msm.extract


@wraps(original_extract)
def monitored_extract(*args, **kwargs):
    if (
        phase == 'comparison'
        and sys._getframe(1).f_code is comparison_api._prepared_view.__code__
    ):
        consume('comparison_extracts')
        assert kwargs.get('selection') == list(range(13))
    return original_extract(*args, **kwargs)


original_compare = dmt.compare_scores


@wraps(original_compare)
def monitored_compare(*args, **kwargs):
    consume('comparison_attempts')
    event('comparison_started', score_name=args[1])
    result = original_compare(*args, **kwargs)
    counts['comparison_returns'] += 1
    # Preserve the actual return before any qualification assertion.
    save(output / f'returned_comparison_{counts["comparison_returns"]}.json', result)
    event('comparison_returned', score_name=args[1])
    assert counts['comparison_returns'] <= budgets['comparison_returns'], (
        'Unexpected extra return preserved'
    )
    return result


replace(dmt, 'prepare_ligand', counted(dmt.prepare_ligand, 'prepare_ligand'))
replace(
    msm.build,
    'assign_partial_charges',
    counted(msm.build.assign_partial_charges, 'partial_charge_assignment'),
)
replace(
    msm.build,
    'assign_autodock_atom_types',
    counted(msm.build.assign_autodock_atom_types, 'atom_type_assignment'),
)
replace(msm.build, 'add_missing_hydrogens', forbidden('hydrogen_generation'))
replace(dmt, 'dock', forbidden('docking'))
replace(dmt, 'score', forbidden('scoring_or_native_energies'))
replace(dmt, 'compare_scores', monitored_compare)
replace(prepared_export, 'export_pdbqt', monitored_export)
replace(comparison_api, 'export_pdbqt', monitored_export)
replace(msm, 'extract', monitored_extract)
replace(
    msm.chemical_states,
    'get_atom_correspondences',
    counted(msm.chemical_states.get_atom_correspondences, 'chemical_correspondences'),
)
replace(
    comparison_api,
    'audit_preparation_charges',
    counted(comparison_api.audit_preparation_charges, 'comparison_charge_audits'),
)
replace(
    comparison_api,
    'audit_pose',
    counted(comparison_api.audit_pose, 'selected_evaluation_audits'),
)
replace(
    scoring_adapter,
    '_input_pose',
    counted(scoring_adapter._input_pose, 'geometry_reads'),
)

In [ ]:
result = dmt.DockingResult.from_dict(deepcopy(collection_record))
source_record = result.to_dict()
assert source_record == collection_record
save(output / 'original_collection.json', source_record)
full_sources, full_before = [], []
prepared_inputs, prepared_before, prepared_native_before = [], [], []
prepared_problems = []
problem_before = []
phase = 'preparation'
for index, (pose, source_path) in enumerate(zip(result, source_paths), 1):
    directory = output / f'pose_{index}'
    directory.mkdir()
    consume('source_loads')
    source = msm.convert(source_path, to_form='molsysmt.MolSys')
    full_sources.append(source)
    full_before.append(native_inspection(source))
    assert msm.get(source, n_atoms=True) == 18
    cut = msm.select(source, selection="atom_name in ['C3','C1']")
    assert len(cut) == len(set(cut)) == 2
    charged = msm.build.assign_partial_charges(source, method='gasteiger_marsili')
    typed = msm.build.assign_autodock_atom_types(
        charged, typing_scheme='autodock4', method='chemical_environment'
    )
    save(
        directory / 'typed18_assignment_inspection.json',
        dict(
            native_charge_unit='elementary_charge',
            native_values=typed.molecular_mechanics.to_dict(),
            charge_assignment=typed.molecular_mechanics.partial_charge_assignment,
            type_assignment=typed.molecular_mechanics.atom_type_assignment,
            scope='Fresh assignments, inspection only; never an MM restoration input',
        ),
    )
    prepared = dmt.prepare_ligand(
        typed,
        selection='all',
        state_id=pose.partner_state_id,
        active_torsion_bonds=[tuple(int(atom) for atom in cut)],
    )
    prepared_inputs.append(prepared)
    prepared_before.append(prepared_fields(prepared))
    prepared_native_before.append(native_inspection(prepared.source_molsys))
    save(
        directory / 'prepared_inspection.json',
        dict(
            prepared=prepared_before[-1],
            native=prepared_native_before[-1],
            scope='New public preparation inspection, not MM persistence or historical restoration',
        ),
    )
    assert prepared.n_atoms == 13 and prepared.torsion_dof == 1
    assert prepared.pdbqt_atom_indices == [6, 7, 8, 9, 10, 11, 12, 0, 1, 2, 3, 4, 5]
    phase = 'gate'
    payload = prepared.to_pdbqt().encode()
    payload_sha = hashlib.sha256(payload).hexdigest()
    assert payload_sha == export_journal[-1]['provider_atom_map']['payload_sha256']
    directory.joinpath('reproduced_ligand.pdbqt').write_bytes(payload)
    for history in pose.metadata['scoring_history']:
        assert history['score_name'] in ('vina', 'vinardo')
        artifacts = history['backend_artifacts']
        dmt.verify_captured_inputs(artifacts)
        assert payload_sha == artifacts['partner']['sha256']
        assert payload == base64.b64decode(
            artifacts['partner']['content_base64'], validate=True
        )
        assert receptor_path.read_bytes() == base64.b64decode(
            artifacts['receptor']['content_base64'], validate=True
        )
    box = pose.score_definitions['vina.lig_inter']['context']['backend_box']
    assert box == {
        'center': contract['backend_box']['center'],
        'size': [24.0, 24.0, 24.0],
        'unit': 'angstrom',
    }
    problem = dmt.DockingProblem(
        receptor=receptor_path,
        partner=prepared,
        search_domain=dmt.BoxRegion(
            puw.quantity(box['center'], 'angstrom'),
            puw.quantity(box['size'], 'angstrom'),
        ),
        metadata={
            'partner_state_id': pose.partner_state_id,
            'receptor_state_id': pose.receptor_state_id,
        },
    )
    prepared_problems.append(problem)
    problem_before.append(problem.to_dict())
    save(directory / 'problem_declaration.json', problem.to_dict())
    save(
        directory / 'captured_input_gate.json',
        dict(
            ligand_sha256=payload_sha,
            receptor_sha256=file_sha(receptor_path),
            matching_scoring_evaluations=['vina', 'vinardo'],
            new_preparation_only=True,
            old_scoring_producer_preserved=True,
        ),
    )
    phase = 'preparation'
assert counts['prepare_ligand'] == counts['partial_charge_assignment'] == 5
assert counts['atom_type_assignment'] == counts['gate_exports'] == 5

In [ ]:
def check_sources():
    assert result.to_dict() == source_record
    assert all(
        p.to_dict() == before for p, before in zip(prepared_problems, problem_before)
    )
    for index, (source, prepared) in enumerate(zip(full_sources, prepared_inputs)):
        assert native_inspection(source) == full_before[index]
        assert prepared_fields(prepared) == prepared_before[index]
        assert (
            native_inspection(prepared.source_molsys) == prepared_native_before[index]
        )


assert puw.configure.report() == caller_policy
profiles = [
    (
        'normal',
        [
            'nm',
            'ps',
            'kelvin',
            'mole',
            'dalton',
            'e',
            'kJ/mol',
            'kJ/(mol*nm)',
            'kJ/(mol*nm**2)',
            'radians',
        ],
    ),
    (
        'pm_fs',
        [
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ],
    ),
]
reports = {}
expected_orders = {'vina': [1, 2, 3, 4, 5], 'vinardo': [1, 2, 4, 3, 5]}
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        active_policy = puw.configure.report()
        for method in ('vina', 'vinardo'):
            phase = 'comparison'
            report = dmt.compare_scores(
                result,
                method + '.lig_inter',
                problems=prepared_problems,
                ascending=True,
            )
            # The wrapper has already preserved the return before these checks.
            save(directory / (method + '_comparison.json'), report)
            assert report['display_order'] == expected_orders[method]
            assert [row['position'] for row in report['rows']] == [1, 2, 3, 4, 5]
            assert len(report['groups']) == len(report['rows']) == 5
            for row, original in zip(report['rows'], source_record['poses']):
                assert row['pose_id'] == original['pose_id']
                assert row['original_rank'] == original['rank']
                assert row['value'] == original['scores'][method + '.lig_inter']
                evidence = row['prepared_evidence']
                assert (
                    evidence['export']['payload_sha256']
                    == row['definition']['context']['partner_sha256']
                )
                assert evidence['chemical_correspondence']['search_complete'] is True
                assert (
                    evidence['original_exported_assignments']['charges']['status']
                    == 'derived'
                )
                assert evidence['validated_charge_assignment']['status'] == 'projected'
                assert evidence['validated_type_assignment']['status'] == 'projected'
            pristine = deepcopy(report)
            consume('json_roundtrips')
            assert json.loads(json.dumps(report, allow_nan=False)) == pristine
            # Change the actual returned report, not a fresh copy of it.
            report['rows'][0]['definition']['context']['weights'][0] = 999
            report['rows'][0]['prepared_evidence']['original_exported_assignments'][
                'charges'
            ]['schema'] = 'detachment-control'
            assert result.to_dict() == source_record
            assert all(
                prepared_fields(p) == before
                for p, before in zip(prepared_inputs, prepared_before)
            )
            consume('detachment_controls')
            report.clear()
            report.update(pristine)
            reports[(profile, method)] = deepcopy(report)
            assert puw.configure.report() == active_policy
    assert puw.configure.report() == caller_policy
phase = 'inspection'
check_sources()
for method in ('vina', 'vinardo'):
    left, right = reports[('normal', method)], reports[('pm_fs', method)]
    assert left['display_order'] == right['display_order']
    assert left['groups'] == right['groups']
    assert [row['value'] for row in left['rows']] == [
        row['value'] for row in right['rows']
    ]
assert counts['comparison_returns'] == 4

In [ ]:
def bad_record(kind):
    record = deepcopy(source_record)
    first = record['poses'][0]
    if kind == 'duplicate_id':
        first['pose_id'] = record['poses'][1]['pose_id']
    elif kind == 'changed_state':
        first['partner_state_id'] = 'other-declared-state'
    elif kind == 'missing_history':
        first['metadata']['scoring_history'] = [
            h for h in first['metadata']['scoring_history'] if h['score_name'] != 'vina'
        ]
    elif kind == 'changed_score':
        first['scores']['vina.lig_inter'] += 0.001
    elif kind == 'changed_shared_weights':
        first['metadata']['score_definitions']['vina.lig_inter']['context']['weights'][
            0
        ] += 0.001
        history = next(
            h for h in first['metadata']['scoring_history'] if h['score_name'] == 'vina'
        )
        history['score_definitions']['vina.lig_inter']['context']['weights'][0] = first[
            'metadata'
        ]['score_definitions']['vina.lig_inter']['context']['weights'][0]
    return record


controls = [
    ('duplicate_id', 'vina.lig_inter', 'unique nonempty pose identifiers'),
    ('changed_state', 'vina.lig_inter', 'same nonempty partner/receptor state'),
    ('total_component', 'vina', 'lig_inter only'),
    ('missing_history', 'vina.lig_inter', 'exactly one recorded scoring evaluation'),
    ('changed_score', 'vina.lig_inter', 'evaluation disagrees'),
    ('changed_shared_weights', 'vina.lig_inter', 'shared comparison context differs'),
]
refusals = []
molecular_counts = (
    'exports',
    'comparison_extracts',
    'chemical_correspondences',
    'comparison_charge_audits',
    'geometry_reads',
)


def refuse(label, candidate, problems, score_name, expected, export_delta=0):
    global phase
    before = candidate.to_dict()
    counters_before = dict(counts)
    phase = 'comparison'
    try:
        dmt.compare_scores(candidate, score_name, problems=problems, ascending=True)
    except CatalogException as error:
        assert expected in str(error), str(error)
        consume('expected_refusals')
        refusal = dict(
            control=label, code=getattr(error, 'code', None), reason=str(error)
        )
        refusals.append(refusal)
        save(output / (label + '_refusal.json'), refusal)
    else:
        raise AssertionError('Expected comparison refusal: ' + label)
    assert candidate.to_dict() == before
    for name in molecular_counts:
        expected_delta = export_delta if name == 'exports' else 0
        assert counts[name] - counters_before[name] == expected_delta, (label, name)


with puw.context(standard_units=profiles[0][1]):
    for label, score_name, expected in controls:
        candidate = dmt.DockingResult.from_dict(bad_record(label))
        refuse(label, candidate, prepared_problems, score_name, expected)
    missing_source = copy(prepared_inputs[0])
    missing_source.source_molsys = None
    missing_problem = dmt.DockingProblem(
        receptor=receptor_path,
        partner=missing_source,
        search_domain=prepared_problems[0].search_domain,
        metadata=deepcopy(prepared_problems[0].metadata),
    )
    refuse(
        'missing_complete_source',
        result,
        [missing_problem, *prepared_problems[1:]],
        'vina.lig_inter',
        'complete molecular source',
    )
    changed_prepared = copy(prepared_inputs[0])
    changed_coordinates = np.array(
        puw.get_value(changed_prepared.coordinates, to_unit='nm'), copy=True
    )
    changed_coordinates[0, 0] += 0.01
    changed_prepared.coordinates = puw.quantity(changed_coordinates, 'nm')
    consume('changed_preparation_controls')
    changed_problem = dmt.DockingProblem(
        receptor=receptor_path,
        partner=changed_prepared,
        search_domain=prepared_problems[0].search_domain,
        metadata=deepcopy(prepared_problems[0].metadata),
    )
    refuse(
        'changed_first_preparation',
        result,
        [changed_problem, *prepared_problems[1:]],
        'vina.lig_inter',
        'prepared export differs',
        export_delta=1,
    )
assert puw.configure.report() == caller_policy
phase = 'inspection'
check_sources()
assert counts == budgets
assert not forbidden_attempts
assert not any(name.split('.')[0] in {'vina', 'molsysviewer'} for name in sys.modules)
for path, expected in stored_inputs.items():
    assert file_sha(Path(path)) == expected
for path, expected in local_inputs_before.items():
    assert file_sha(Path(path)) == expected
assert (
    file_sha(Path(comparison_api.__file__)) == identities['comparison_source']['sha256']
)
assert file_sha(native_extension) == identities['native_extension']['sha256']
assignment_modules = {}
for name in ('rdkit', 'rdkit.Chem.rdchem', 'rdkit.Chem.rdPartialCharges'):
    module = sys.modules[name]
    path = Path(module.__file__).resolve()
    assignment_modules[name] = dict(path=str(path), sha256=file_sha(path))
save(output / 'assignment_provider_identity.json', assignment_modules)
save(output / 'refusals.json', refusals)
save(
    output / 'receipt.json',
    dict(
        schema='bindingmt.e1_public_comparison_receiving@1',
        owner='uibcdf/bindingmt#78',
        counts=counts,
        trusted_case_roots=ROOTS,
        runtime=identities,
        newly_prepared_inputs=5,
        new_charge_assignments=5,
        new_type_assignments=5,
        new_hydrogens=0,
        new_energies=0,
        new_searches=0,
        positive_comparisons=4,
        methods=['vina', 'vinardo'],
        reader_policy_repetitions=2,
        distinct_scoring_populations=1,
        expected_orders=expected_orders,
        all_original_records_preserved=True,
        full_native_inspection_scope='Public topology/chemical-state/structure snapshots and separate MM values/reports; no MM codec',
        comparison_scope='Current complete prepared views bound to existing captured scores; no producer authentication or historical preparation restoration',
        original_result_multi_pose_continuity_received=False,
        full_e1_accepted=False,
    ),
)
for owner, name, original in reversed(patches):
    setattr(owner, name, original)
sys.meta_path.remove(import_guard)
print(json.dumps(dict(outcome='PASS', counts=counts, new_empirical_values=0)))